# Full-Scale Upgrade Guide: 50 Clusters → 1,200+ Clusters

This notebook documents every change needed across the codebase when scaling
from the 50-cluster sample to the full 1,247 nationwide DHS clusters.

## Summary of changes per phase

| Phase | File | Change Type |
|---|---|---|
| 1a | phase1-sentinel-training-images.ipynb | Path only |
| 1b | phase1-ntl-median.ipynb | Path + column name |
| 1c | phase1-proxy-cnn.ipynb | Path + batch size + epochs |
| 2 | phase2-feature-extraction.ipynb | Path |
| 3 | phase3-osm-feature-extraction.ipynb | Path + output name |
| 4 | phas4-final-model.ipynb | Paths + hyperparams + K-Fold |
| 5 | phase5-shap-analysis.ipynb | Paths + background size |

---
## PHASE 1a — Sentinel Training Images
**File:** `phase1-sentinel-training-images.ipynb`

No code changes needed. GEE already exports all 1,247 clusters because it
iterates over the full `PH_DHS_GPS` asset. Just re-run with a full Drive
and sufficient GEE quota. The skip-logic (`os.path.exists`) means already-
downloaded tiles are not re-exported.

---
## PHASE 1b — NTL Median (VIIRS Labels)
**File:** `phase1-ntl-median.ipynb`

Change the output filename from the sample version to the full version.

In [ ]:
# ---- BEFORE (sample) ----
# output_csv = "viirs_ntl_labels_sample50.csv"

# ---- AFTER (full scale) ----
output_csv = "viirs_ntl_labels_full.csv"

# Also ensure the NTL_Class column uses the same 3-class scheme:
# 0 = Dark  (radiance < 1.0 nW/cm²/sr)
# 1 = Dim   (1.0 <= radiance < 10.0)
# 2 = Bright (radiance >= 10.0)
# Adjust thresholds based on your VIIRS data distribution.

---
## PHASE 1c — Proxy CNN Training
**File:** `phase1-proxy-cnn.ipynb`

Three changes:
1. Update `tif_folder` and `labels_file` paths
2. Reduce batch size to avoid memory issues with 5,000 images
3. Add EarlyStopping to avoid overfitting with more data

In [ ]:
# ---- BEFORE (sample) ----
# tif_folder   = ".../sample50-quarterly-2022"
# labels_file  = "viirs_ntl_labels_sample50.csv"

# ---- AFTER (full scale) ----
tif_folder  = "/path/to/full-quarterly-2022"   # All 1247 cluster folders
labels_file = "viirs_ntl_labels_full.csv"

# ---- Training call changes ----
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

callbacks = [
    EarlyStopping(
        monitor   = 'val_accuracy',
        patience  = 5,
        restore_best_weights = True
    ),
    ModelCheckpoint(
        'cnn_viirs_proxy_best.keras',
        monitor   = 'val_accuracy',
        save_best_only = True
    )
]

history = model.fit(
    train_gen,
    validation_data = val_gen,
    epochs          = 30,     # More epochs; EarlyStopping prevents over-training
    callbacks       = callbacks,
    verbose         = 1
)

# Save in native Keras format (avoids HDF5 deprecation warning)
model.save('cnn_viirs_proxy.keras')

---
## PHASE 2 — Feature Extraction
**File:** `phase2-feature-extraction.ipynb`

In [ ]:
# ---- BEFORE (sample) ----
# tif_folder  = ".../sample50-quarterly-2022"
# model_path  = "cnn_viirs_proxy.h5"
# output_csv  = "dynamic_features_per_quarter.csv"

# ---- AFTER (full scale) ----
tif_folder = "/path/to/full-quarterly-2022"
model_path = "cnn_viirs_proxy.keras"   # or .h5 if you kept the old format
output_csv = "dynamic_features_full.csv"

# PERFORMANCE TIP for 5000+ images:
# Process in batches and save incrementally to avoid OOM errors
BATCH_SIZE  = 16
SAVE_EVERY  = 500   # save checkpoint CSV every N images

# Replace the single-image predict loop with batch predict:
# (pseudocode — integrate into the existing extraction loop)
#
# batch_imgs = []
# batch_meta = []
# for each image:
#     batch_imgs.append(img)
#     batch_meta.append((cluster_id, quarter))
#     if len(batch_imgs) == BATCH_SIZE:
#         features = feature_extractor.predict(np.array(batch_imgs), verbose=0)
#         ... store results ...
#         batch_imgs, batch_meta = [], []

---
## PHASE 3 — OSM Feature Extraction
**File:** `phase3-osm-feature-extraction.ipynb`

In [ ]:
# ---- BEFORE (sample) ----
# viirs_folder_path = ".../VIIRS-PhilSA-50Sample"
# output_csv        = "static_osm_features_detailed.csv"

# ---- AFTER (full scale) ----
viirs_folder_path = "/path/to/VIIRS-PhilSA-Full"  # Full VIIRS rasters
output_csv        = "static_osm_features_full.csv"

# PERFORMANCE TIP: the spatial join loop is the bottleneck at 1247 clusters.
# Consider chunking by region and saving partial CSVs:
#
# REGIONS = gdf_dhs['DHSREG'].unique()
# for region in REGIONS:
#     gdf_subset = gdf_dhs[gdf_dhs['DHSREG'] == region]
#     ... run feature extraction on subset ...
#     partial_df.to_csv(f'osm_partial_{region}.csv', index=False)
#
# Then concatenate:
# import glob
# pd.concat([pd.read_csv(f) for f in glob.glob('osm_partial_*.csv')]).to_csv(output_csv)

---
## PHASE 4 — Final Hybrid Model (Major Changes)
**File:** `phas4-final-model.ipynb`

This is the most significant upgrade. Changes:
1. Update input file paths
2. Switch from train/test split to K-Fold Cross Validation
3. Increase LSTM units (more data = more capacity)
4. Add EarlyStopping
5. Report all evaluation metrics: R², RMSE, Pearson r

In [ ]:
# ==========================================
# 1. UPDATE PATHS
# ==========================================

# ---- BEFORE (sample) ----
# path_dynamic = "dynamic_features_per_quarter.csv"
# path_static  = "static_osm_features_sample50.csv"
# path_dhs     = "dhs_wealth_sample50.csv"

# ---- AFTER (full scale) ----
path_dynamic = "dynamic_features_full.csv"
path_static  = "static_osm_features_full.csv"
path_dhs     = "dhs_wealth.csv"              # All 1247 clusters

In [ ]:
# ==========================================
# 2. UPGRADED MODEL ARCHITECTURE
# ==========================================
# With 1247 clusters the model can support more capacity.

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Concatenate, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

def build_model(n_static_features):
    # Branch 1: LSTM for temporal satellite features
    input_dynamic = Input(shape=(4, 4096), name='Dynamic_Input')
    x1 = LSTM(
        64,                          # Increased from 16 → 64
        return_sequences = False,
        kernel_regularizer = l2(1e-4)
    )(input_dynamic)
    x1 = Dropout(0.4)(x1)

    # Branch 2: Dense for static OSM + VIIRS features
    input_static = Input(shape=(n_static_features,), name='Static_Input')
    x2 = Dense(32, activation='relu', kernel_regularizer=l2(1e-4))(input_static)
    x2 = BatchNormalization()(x2)
    x2 = Dropout(0.3)(x2)

    # Fusion
    combined = Concatenate()([x1, x2])
    z = Dense(32, activation='relu', kernel_regularizer=l2(1e-4))(combined)
    z = Dropout(0.3)(z)
    output = Dense(1, activation='linear', name='Wealth_Prediction')(z)

    model = Model(inputs=[input_dynamic, input_static], outputs=output)
    model.compile(
        optimizer = Adam(learning_rate=0.001),
        loss      = 'mse',
        metrics   = ['mae']
    )
    return model

In [ ]:
# ==========================================
# 3. K-FOLD CROSS VALIDATION
# ==========================================
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
from scipy.stats import pearsonr
import numpy as np

N_FOLDS = 5
kf      = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED_VALUE)

fold_metrics = []
all_y_true   = []
all_y_pred   = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X_dynamic)):
    print(f"\n{'='*40}")
    print(f"FOLD {fold+1}/{N_FOLDS}")
    print(f"  Train: {len(train_idx)} | Val: {len(val_idx)}")

    # Split
    X_dyn_tr, X_dyn_val   = X_dynamic[train_idx], X_dynamic[val_idx]
    X_stat_tr, X_stat_val = X_static[train_idx],  X_static[val_idx]
    y_tr, y_val           = y[train_idx],          y[val_idx]

    # Scale static features (fit only on training fold)
    scaler = StandardScaler()
    X_stat_tr  = scaler.fit_transform(X_stat_tr)
    X_stat_val = scaler.transform(X_stat_val)

    # Build fresh model each fold
    model = build_model(X_static.shape[1])

    callbacks = [
        EarlyStopping(
            monitor              = 'val_loss',
            patience             = 15,
            restore_best_weights = True
        ),
        ReduceLROnPlateau(
            monitor  = 'val_loss',
            factor   = 0.5,
            patience = 7,
            min_lr   = 1e-6
        )
    ]

    history = model.fit(
        x               = [X_dyn_tr, X_stat_tr],
        y               = y_tr,
        validation_data = ([X_dyn_val, X_stat_val], y_val),
        epochs          = 200,
        batch_size      = 32,
        callbacks       = callbacks,
        verbose         = 0
    )

    # Evaluate
    preds    = model.predict([X_dyn_val, X_stat_val], verbose=0).flatten()
    r2       = r2_score(y_val, preds)
    rmse     = np.sqrt(np.mean((y_val - preds) ** 2))
    pearson  = pearsonr(y_val, preds)[0]
    best_ep  = np.argmin(history.history['val_loss']) + 1

    fold_metrics.append({'fold': fold+1, 'R2': r2, 'RMSE': rmse, 'Pearson_r': pearson})
    all_y_true.extend(y_val.tolist())
    all_y_pred.extend(preds.tolist())

    print(f"  R²={r2:.4f}  RMSE={rmse:.4f}  r={pearson:.4f}  Best epoch={best_ep}")

# --- Summary ---
df_metrics = pd.DataFrame(fold_metrics)
print("\nK-Fold Cross Validation Results:")
print(df_metrics.to_string(index=False))
print(f"\nMean R²  : {df_metrics['R2'].mean():.4f} ± {df_metrics['R2'].std():.4f}")
print(f"Mean RMSE: {df_metrics['RMSE'].mean():.4f} ± {df_metrics['RMSE'].std():.4f}")
print(f"Mean r   : {df_metrics['Pearson_r'].mean():.4f} ± {df_metrics['Pearson_r'].std():.4f}")

# Overall (pooled) metrics
y_true_arr = np.array(all_y_true)
y_pred_arr = np.array(all_y_pred)
print(f"\nPooled R²  : {r2_score(y_true_arr, y_pred_arr):.4f}")
print(f"Pooled RMSE: {np.sqrt(np.mean((y_true_arr - y_pred_arr)**2)):.4f}")
print(f"Pooled r   : {pearsonr(y_true_arr, y_pred_arr)[0]:.4f}")

In [ ]:
# ==========================================
# 4. TRAIN FINAL MODEL ON ALL DATA
# ==========================================
# After K-Fold validation confirms performance, train one final model
# on the entire dataset for use in inference and SHAP.

print("Training final model on all data...")
scaler_final    = StandardScaler()
X_static_scaled = scaler_final.fit_transform(X_static)

final_model = build_model(X_static.shape[1])

final_history = final_model.fit(
    x          = [X_dynamic, X_static_scaled],
    y          = y,
    epochs     = 100,           # Fixed epochs (no val split, so no EarlyStopping)
    batch_size = 32,
    verbose    = 1
)

final_model.save('final_hybrid_poverty_model.keras')

import pickle
with open('final_scaler.pkl', 'wb') as f:
    pickle.dump(scaler_final, f)

print("Saved: final_hybrid_poverty_model.keras")
print("Saved: final_scaler.pkl  (needed for inference + SHAP)")

---
## PHASE 5 — SHAP Analysis
**File:** `phase5-shap-analysis.ipynb`

Two changes for the full dataset:

In [ ]:
# ==========================================
# UPDATE IN phase5-shap-analysis.ipynb
# ==========================================

# ---- BEFORE (sample) ----
# path_dynamic = "dynamic_features_per_quarter.csv"
# path_static  = "static_osm_features_sample50.csv"
# path_dhs     = "dhs_wealth_sample50.csv"
# path_model   = "final_hybrid_poverty_model.h5"
# N_BACKGROUND = min(100, len(X_shap))

# ---- AFTER (full scale) ----
path_dynamic = "dynamic_features_full.csv"
path_static  = "static_osm_features_full.csv"
path_dhs     = "dhs_wealth.csv"
path_model   = "final_hybrid_poverty_model.keras"

# Also load the scaler saved in Phase 4
import pickle
with open('final_scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)
# Replace: scaler.fit(X_stat_train)  →  (already fitted, just use scaler.transform)

N_BACKGROUND = 200   # Increase from 100 for better SHAP stability with 1247 clusters

# SHAP computation for 1247 clusters with N_BACKGROUND=200, nsamples=100
# Estimated time on M4 MacBook Air (parallelised): ~2-6 hours
# Run as a batch job or overnight.

---
## Migration Checklist

Work through these steps in order:

- [ ] GEE export: queue all 1247 cluster exports (phase1a). Confirm ~6000 TIFs in Drive.
- [ ] NTL labels: run phase1b on full VIIRS data. Output: `viirs_ntl_labels_full.csv`
- [ ] Proxy CNN: update paths, add EarlyStopping, retrain. Output: `cnn_viirs_proxy.keras`
- [ ] Feature extraction: update paths, use batch predict. Output: `dynamic_features_full.csv`
- [ ] OSM features: update paths. Output: `static_osm_features_full.csv`
- [ ] DHS wealth: confirm `dhs_wealth.csv` has all 1247 clusters (already done in phase4-wealth-index.ipynb)
- [ ] Final model: run K-Fold, train final model. Output: `final_hybrid_poverty_model.keras`, `final_scaler.pkl`
- [ ] SHAP: update paths + N_BACKGROUND=200. Run overnight. Output: `shap_outputs/`
- [ ] Integrate `shap_outputs/` into Next.js dashboard